# Hello World — Survival Analysis Pipeline

Self-contained pipeline using `train.csv` and `val_test.csv`.

- `last_observed_age` is derived from `max(Age_v1, ..., Age_v22)` columns in `train.csv`
- Evaluation via `sksurv.metrics.concordance_index_censored`

## 1. Imports & Config

In [86]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV

TRAIN_PATH    = 'data/processed_wide/train.csv'
TEST_PATH = 'data/processed_wide/val_test.csv'
OUTPUT_PATH   = 'hello_world_submission.csv'

# Columns that must NOT be used as features
TARGET_COLS = [
    'evenements_hepatiques_majeurs',
    'evenements_hepatiques_age_occur',
    'death',
    'death_age_occur',
]
ID_COLS = ['patient_id_anon', 'trustii_id']

# Maximum fraction of missing values allowed for a feature to be used.
# 180/261 features have >80% missing in train; with median imputation they
# collapse to a constant and the model memorises the imputed-zero pattern.
MAX_MISSING_RATE = 0.50

print('Imports OK')

Imports OK


## 2. Load Data & Derive `last_observed_age`

In [87]:
train_df    = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'train shape    : {train_df.shape}')
print(f'test shape : {test_df.shape}')

# Derive last_observed_age from Age_v1 .. Age_v22 columns.
# These columns record patient age at each follow-up visit.
# The maximum across visits gives the age at last observation (censoring time reference).
age_cols = [c for c in train_df.columns if c.startswith('Age_v')]
assert len(age_cols) > 0, 'No Age_v* columns found — re-run data_prep_final.ipynb'

train_df['last_observed_age'] = train_df[age_cols].max(axis=1)

print(f'\nAge_v* columns used ({len(age_cols)}): {age_cols[:5]} ...')
print(f'last_observed_age stats:\n{train_df["last_observed_age"].describe()}')

train shape    : (1253, 287)
test shape : (423, 284)

Age_v* columns used (22): ['Age_v1', 'Age_v2', 'Age_v3', 'Age_v4', 'Age_v5'] ...
last_observed_age stats:
count    1253.000000
mean       59.719074
std        12.974573
min        19.000000
25%        52.000000
50%        62.000000
75%        69.000000
max        89.000000
Name: last_observed_age, dtype: float64


## 3. Target Processing

`prepare_survival_targets()` converts the 4 raw target columns in `train.csv` into a `sksurv`-compatible structured array.

### Survival time formula
- **Event patients** : `age_at_event − Age_v1` (time from first visit to event)
- **Censored patients** : `last_observed_age − Age_v1` (time from first visit to last follow-up)

### Filtering
- **Hepatic** : drop rows where event=1 but `age_occur` is NaN
- **Death** : drop rows where `death` is NaN (unknown outcome) or event=1 but `age_occur` is NaN

In [88]:
def prepare_survival_targets(df, outcome='hepatic'):
    """
    Build a sksurv-compatible structured array from the 4 target columns in train.csv.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain:
          - 'Age_v1'            : patient age at first visit (baseline)
          - 'last_observed_age' : max(Age_v1..Age_v22) — age at last follow-up
          - 4 target columns    : see TARGET_COLS global
    outcome : str
        'hepatic' — predict major hepatic events
        'death'   — predict all-cause mortality

    Returns
    -------
    df_valid : pd.DataFrame
        Filtered rows (invalid/unknown outcomes removed), index reset.
    y : structured np.ndarray
        sksurv structured array with fields (event: bool, time: float).
        Compatible with concordance_index_censored and all sksurv estimators.

    Survival time formula
    ---------------------
      event patients  -> age_at_event - Age_v1
      censored        -> last_observed_age - Age_v1
    """
    df = df.copy()

    if outcome == 'hepatic':
        event_col     = 'evenements_hepatiques_majeurs'
        age_occur_col = 'evenements_hepatiques_age_occur'
        name          = 'Hepatic_event'
        is_event = df[event_col] == 1
        invalid  = is_event & df[age_occur_col].isna()
        mask     = ~invalid

    elif outcome == 'death':
        event_col     = 'death'
        age_occur_col = 'death_age_occur'
        name          = 'Death'
        is_event = df[event_col] == 1
        unknown  = df[event_col].isna()
        invalid  = is_event & df[age_occur_col].isna()
        mask     = ~unknown & ~invalid

    else:
        raise ValueError(f"outcome must be 'hepatic' or 'death', got {outcome!r}")

    df_valid   = df[mask].copy().reset_index(drop=True)
    is_event_v = (df_valid[event_col] == 1)

    # Survival time (in same age-year units as the dataset)
    time_values = np.where(
        is_event_v,
        df_valid[age_occur_col] - df_valid['Age_v1'],       # event patients
        df_valid['last_observed_age'] - df_valid['Age_v1'], # censored patients
    ).astype(float)

    # Clamp to small positive value (sksurv requires strictly positive times)
    time_values = np.maximum(time_values, 0.001)

    y = Surv.from_arrays(
        event=is_event_v.astype(bool).values,
        time=time_values,
        name_event=name,
        name_time='Time_years',
    )
    return df_valid, y


# --- Sanity check ---
df_hep,   y_hep   = prepare_survival_targets(train_df, outcome='hepatic')
df_death, y_death = prepare_survival_targets(train_df, outcome='death')

print('Hepatic model')
print(f'  Patients : {len(df_hep)}')
print(f'  Events   : {y_hep["Hepatic_event"].sum()} ({100*y_hep["Hepatic_event"].mean():.1f}%)')
print(f'  Time     : min={y_hep["Time_years"].min():.2f}, max={y_hep["Time_years"].max():.2f}')
print()
print('Death model')
print(f'  Patients : {len(df_death)}')
print(f'  Events   : {y_death["Death"].sum()} ({100*y_death["Death"].mean():.1f}%)')
print(f'  Time     : min={y_death["Time_years"].min():.2f}, max={y_death["Time_years"].max():.2f}')

Hepatic model
  Patients : 1253
  Events   : 47 (3.8%)
  Time     : min=0.00, max=21.00

Death model
  Patients : 984
  Events   : 76 (7.7%)
  Time     : min=0.00, max=21.00


## 4. Feature Matrix

In [89]:
def build_feature_matrix(df, keep_cols=None):
    """
    Build numeric feature matrix, excluding:
      - Target columns (would cause leakage)
      - ID columns
      - Age_v* columns (used only for censoring time derivation)
      - 'last_observed_age' (derived from Age_v* — leakage risk)
    If keep_cols is provided, only those columns are returned (for
    consistent alignment across train / val / test).
    """
    age_v_cols   = [c for c in df.columns if c.startswith('Age_v')]
    leakage_cols = ['last_observed_age']
    drop_cols    = TARGET_COLS + ID_COLS + age_v_cols + leakage_cols
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    X = X.select_dtypes(include='number')
    if keep_cols is not None:
        X = X[[c for c in keep_cols if c in X.columns]]
    return X


# ── Build raw matrices ────────────────────────────────────────────────────
X_hep_raw   = build_feature_matrix(df_hep)
X_death_raw = build_feature_matrix(df_death)
X_pred_raw  = build_feature_matrix(test_df)

print(f'Raw feature count: {X_hep_raw.shape[1]}')

# ── Missing-rate filter (fitted on train, applied everywhere) ─────────────
# Features with > MAX_MISSING_RATE missing in train collapse to a constant
# after median imputation → the model memorises the imputed pattern instead
# of a real signal (a form of overfitting).  Removing them first dramatically
# reduces feature/event ratio (261 → ~36) which is the primary cause of the
# train/val gap.
missing_rate_hep   = X_hep_raw.isna().mean()
missing_rate_death = X_death_raw.isna().mean()

keep_hep   = missing_rate_hep[missing_rate_hep   <= MAX_MISSING_RATE].index.tolist()
keep_death = missing_rate_death[missing_rate_death <= MAX_MISSING_RATE].index.tolist()

# Align to val_test (prediction set must have the same columns)
keep_hep   = [c for c in keep_hep   if c in X_pred_raw.columns]
keep_death = [c for c in keep_death if c in X_pred_raw.columns]

# Final aligned matrices
X_hep_aln   = X_hep_raw[keep_hep]
X_death_aln = X_death_raw[keep_death]
X_pred_hep  = X_pred_raw[keep_hep]
X_pred_death= X_pred_raw[keep_death]

print(f'Hepatic features after missing filter : {len(keep_hep)}  '
      f'(EPV = {int((df_hep["evenements_hepatiques_majeurs"]==1).sum())}/{len(keep_hep)} = '
      f'{(df_hep["evenements_hepatiques_majeurs"]==1).sum()/len(keep_hep):.2f})')
print(f'Death   features after missing filter : {len(keep_death)}  '
      f'(EPV = {int((df_death["death"]==1).sum())}/{len(keep_death)} = '
      f'{(df_death["death"]==1).sum()/len(keep_death):.2f})')


Raw feature count: 260
Hepatic features after missing filter : 35  (EPV = 47/35 = 1.34)
Death   features after missing filter : 44  (EPV = 76/44 = 1.73)


## 5. Train Models & Evaluate

Two model types per outcome:
- **Elastic Net Cox** (`CoxnetSurvivalAnalysis`, l1_ratio=0.9) — LASSO-style sparsity, alpha tuned by 5-fold CV
- **Random Survival Forest** — `min_samples_leaf=20` to reduce overfitting on rare events (47 events / ~36 features)

> **Why the old models overfit**: 261 features × 47 events = EPV 0.18 (need ≥ 10). After the missing-rate filter the
> effective feature count drops to ~36 (EPV ≈ 1.3), and the elastic net further shrinks to ≤ 10 active coefficients.


In [90]:
# ── Elastic Net Cox: fit alpha path, pick best alpha with 5-fold CV ──────
def fit_coxnet(X, y, l1_ratio=0.9, n_splits=5, random_state=42):
    """
    Fit an Elastic-Net regularised Cox PH model.
    Alpha is selected by 5-fold cross-validation over the full regularisation
    path, which automatically enforces EPV constraints via sparsity.
    Returns a fitted Pipeline (imputer → scaler → CoxnetSurvivalAnalysis).
    """
    # Step 1: estimate the regularisation path
    pipe_path = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('cox', CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alpha_min_ratio=0.01,
                                       max_iter=1000, fit_baseline_model=True)),
    ])
    pipe_path.fit(X, y)
    alphas = pipe_path.named_steps['cox'].alphas_

    # Step 2: cross-validate to find best alpha
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    gcv = GridSearchCV(
        Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler()),
            ('cox', CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, max_iter=1000,
                                           fit_baseline_model=True)),
        ]),
        param_grid={'cox__alphas': [[a] for a in alphas]},
        cv=cv, error_score=0.5, n_jobs=-1,
    )
    gcv.fit(X, y)
    return gcv.best_estimator_


def make_rsf_pipeline():
    """RSF with min_samples_leaf=20 to reduce overfitting on rare events."""
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('rsf', RandomSurvivalForest(
            n_estimators=300,
            min_samples_leaf=20,   # was 10 — larger leaves → less variance
            min_samples_split=40,  # require more samples before splitting
            max_features='sqrt',
            n_jobs=-1,
            random_state=42,
        )),
    ])


# ── Hepatic models ────────────────────────────────────────────────────────
print('Fitting Hepatic — Elastic Net Cox (CV over alpha path)...')
cox_hep = fit_coxnet(X_hep_aln, y_hep)
best_alpha_hep = cox_hep.named_steps['cox'].alphas_[0]
n_nonzero_hep  = (cox_hep.named_steps['cox'].coef_ != 0).sum()

print('Fitting Hepatic — RSF...')
rsf_hep = make_rsf_pipeline()
rsf_hep.fit(X_hep_aln, y_hep)

ci_cox_hep = concordance_index_censored(
    y_hep['Hepatic_event'], y_hep['Time_years'], cox_hep.predict(X_hep_aln))[0]
ci_rsf_hep = concordance_index_censored(
    y_hep['Hepatic_event'], y_hep['Time_years'], rsf_hep.predict(X_hep_aln))[0]

print(f'\n=== Hepatic Event Model ===')
print(f'  Elastic Net Cox  alpha={best_alpha_hep:.4f}  active coefs={n_nonzero_hep}  C-index (train): {ci_cox_hep:.4f}')
print(f'  RSF                                                      C-index (train): {ci_rsf_hep:.4f}')


# ── Death models ──────────────────────────────────────────────────────────
print('\nFitting Death — Elastic Net Cox (CV over alpha path)...')
cox_death = fit_coxnet(X_death_aln, y_death)
best_alpha_death = cox_death.named_steps['cox'].alphas_[0]
n_nonzero_death  = (cox_death.named_steps['cox'].coef_ != 0).sum()

print('Fitting Death — RSF...')
rsf_death = make_rsf_pipeline()
rsf_death.fit(X_death_aln, y_death)

ci_cox_death = concordance_index_censored(
    y_death['Death'], y_death['Time_years'], cox_death.predict(X_death_aln))[0]
ci_rsf_death = concordance_index_censored(
    y_death['Death'], y_death['Time_years'], rsf_death.predict(X_death_aln))[0]

print(f'\n=== Death Model ===')
print(f'  Elastic Net Cox  alpha={best_alpha_death:.4f}  active coefs={n_nonzero_death}  C-index (train): {ci_cox_death:.4f}')
print(f'  RSF                                                      C-index (train): {ci_rsf_death:.4f}')


Fitting Hepatic — Elastic Net Cox (CV over alpha path)...
Fitting Hepatic — RSF...

=== Hepatic Event Model ===
  Elastic Net Cox  alpha=0.0196  active coefs=5  C-index (train): 0.7693
  RSF                                                      C-index (train): 0.9118

Fitting Death — Elastic Net Cox (CV over alpha path)...
Fitting Death — RSF...

=== Death Model ===
  Elastic Net Cox  alpha=0.0486  active coefs=2  C-index (train): 0.7183
  RSF                                                      C-index (train): 0.9344


## 6. Predictions on `val_test.csv`

In [92]:
# Use RSF predictions (more robust on tabular survival data with rare events)
pred_hep   = rsf_hep.predict(X_pred_hep)
pred_death = rsf_death.predict(X_pred_death)

submission = pd.DataFrame({
    'trustii_id':         test_df['trustii_id'].values,
    'risk_hepatic_event': pred_hep,
    'risk_death':         pred_death,
})

submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(submission)} predictions → {OUTPUT_PATH}')
print(submission.head())


Saved 423 predictions → hello_world_submission.csv
   trustii_id  risk_hepatic_event  risk_death
0           1            0.593542    2.103396
1           2            0.442713    2.069411
2           3            0.216856    2.634236
3           4            0.477418    5.398052
4           5            1.355168    1.971027
